# 03 · Gomory cuts from the optimal basis
**Branch-and-Cut Workshop · UANL & Tecnológico de Monterrey** (Prof. Angélica Salazar, Prof. Vincent Boyer)

*Activity 4 & 5 · about 13 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you. Cells marked **🤖 AI prompt** contain a prompt you can paste into
> Gemini (the ✨ button in Colab), ChatGPT, Claude… **Always read and test the code the AI gives you.**

**Recipe.** Take the optimal basis of the LP relaxation. Each row of $B^{-1}[A \mid I]\,x = B^{-1}b$ reads
$$x_{B(i)} + \sum_{j \in N} \bar a_{ij}\, x_j = \bar b_i .$$
If $\bar b_i$ is fractional, the **Gomory fractional cut** is
$$\sum_{j\in N} f_j\, x_j \ge f_0, \qquad f_j = \text{frac}(\bar a_{ij}),\; f_0 = \text{frac}(\bar b_i),$$
valid for every integer point (all variables, slacks included, are integer when $A, b$ are integer).
At the LP optimum all non-basic $x_j$ are 0, so the cut is violated: the fractional point is **cut off**.

This notebook takes **any small model written in PuLP**, finds the last basis, and prints every equation with
fractions. Add the cut, re-solve, repeat until the LP solution is integer.

In [ ]:
# @title Setup and helper functions (run this cell; no need to read it)
# PuLP 4 with the open-source HiGHS solver (not pre-installed on Colab: installed on first run, ~20 s)
try:
    import pulp, highspy
    if int(pulp.__version__.split(".")[0]) < 4:
        raise ImportError("old PuLP")
except ImportError:
    import importlib, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[highs]>=4"], check=True)
    for _m in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[_m]                       # forget an older PuLP loaded earlier in this session
    importlib.invalidate_caches()
    import pulp
SOLVER = pulp.HiGHS(msg=False)                    # every model in these notebooks is solved with HiGHS
print("PuLP", pulp.__version__, "+ HiGHS")


import math, re
from fractions import Fraction as Fr
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

def lp_data(prob):
    """Read  max c x  s.t.  A x <= b,  x >= 0  from a PuLP model, in exact fractions.
    '>=' rows are multiplied by -1; a 'min' objective is turned into 'max -c x'."""
    xs = prob.variables()
    if (not xs or not prob.objective or all(prob.objective.get(v, 0) == 0 for v in xs)
            or all(con.get(v, 0) == 0 for con in prob.constraints() for v in xs)):
        raise ValueError(f"Model '{prob.name}' has no variables or no constraints with non-zero coefficients. "
                         "Did you fill in the ✍️ TODO lines (objective and constraints)?")
    q = lambda v: Fr(v).limit_denominator(10**6)
    sign = 1 if prob.sense == pulp.LpMaximize else -1
    c = [sign * q(prob.objective.get(v, 0)) for v in xs]
    A, b = [], []
    for con in prob.constraints():
        row, rhs = [q(con.get(v, 0)) for v in xs], -q(con.constant)
        if con.sense == pulp.LpConstraintGE:
            row, rhs = [-a for a in row], -rhs
        elif con.sense == pulp.LpConstraintEQ:
            raise ValueError("this small demo handles <= and >= constraints only")
        A.append(row); b.append(rhs)
    return [v.name for v in xs], c, A, b

def matmul(X, Y):
    return [[sum(X[i][k] * Y[k][j] for k in range(len(Y))) for j in range(len(Y[0]))] for i in range(len(X))]

def nice(v, den=1000):
    """Float from the solver -> short fraction string, e.g. 3.3333333 -> '10/3'."""
    return str(Fr(v).limit_denominator(den))

# ---- LaTeX output (fractions shown as fractions) ----
def tex_num(q):
    q = Fr(q)
    if q.denominator == 1:
        return str(q.numerator)
    s = r"\tfrac{%d}{%d}" % (abs(q.numerator), q.denominator)
    return "-" + s if q < 0 else s

def tex_var(name):
    return re.sub(r"^([A-Za-z]+)_?(\d+)$", r"\1_{\2}", name)

def tex_mat(M):
    return r"\begin{pmatrix}" + r"\\ ".join(" & ".join(tex_num(v) for v in row) for row in M) + r"\end{pmatrix}"

def tex_col(v):
    return tex_mat([[a] for a in v])

def tex_expr(coefs, names):
    out = ""
    for a, nm in zip(coefs, names):
        a = Fr(a)
        if a == 0:
            continue
        sign = ("-" if a < 0 else "") if not out else (" - " if a < 0 else " + ")
        out += sign + ("" if abs(a) == 1 else tex_num(abs(a)) + " ") + tex_var(nm)
    return out or "0"

def show(*parts):
    display(Math(r"\qquad ".join(parts)))

def plot_model(prob, path=(), cuts=(), points=True, star=None, ax=None, title=""):
    """Feasible region of a 2-variable model, its integer points, a simplex path and cuts (a, beta, label)."""
    names, c, A, b = lp_data(prob)
    A, b = np.array(A, float), np.array(b, float)
    hi = 1.15 * max(b[i] / A[i, j] for i in range(len(b)) for j in range(2) if A[i, j] > 0)
    ax = ax or plt.subplots(figsize=(5.5, 4.5))[1]
    X, Y = np.meshgrid(np.linspace(0, hi, 500), np.linspace(0, hi, 500))
    ok = np.all([A[i, 0] * X + A[i, 1] * Y <= b[i] + 1e-9 for i in range(len(b))], axis=0)
    ax.contourf(X, Y, ok, levels=[0.5, 1], colors=["#0072CE"], alpha=0.15)
    xs = np.linspace(0, hi, 200)
    for i in range(len(b)):
        if A[i, 1] != 0:
            ax.plot(xs, (b[i] - A[i, 0] * xs) / A[i, 1], color="#003865", lw=1)
    if points:
        pts = [(i, j) for i in range(int(hi) + 1) for j in range(int(hi) + 1)
               if all(A[k, 0] * i + A[k, 1] * j <= b[k] + 1e-9 for k in range(len(b)))]
        ax.scatter(*zip(*pts), s=14, color="k", zorder=3)
    for k, (a, beta, label) in enumerate(cuts):
        a = [float(v) for v in a]
        ax.plot(xs, (float(beta) - a[0] * xs) / a[1], "--", lw=2, label=label)
    if len(path):
        P = np.array([[float(v) for v in p] for p in path])
        ax.plot(P[:, 0], P[:, 1], "-o", color="#E07B00", lw=2, zorder=4, label="simplex path")
    if star is not None:
        ax.scatter([float(star[0])], [float(star[1])], marker="*", s=220, color="#C8102E", zorder=5)
    ax.set_xlim(-0.2, hi); ax.set_ylim(-0.2, hi); ax.set_xlabel(names[0]); ax.set_ylabel(names[1])
    ax.set_title(title or prob.name)
    if cuts or len(path):
        ax.legend(fontsize=8, loc="upper right")
    return ax

def simplex_steps(prob, verbose=True):
    """Revised simplex on the LP relaxation of prob, in exact fractions.
    Starts from the slack basis (x = 0); the variable with the largest reduced cost enters.
    At every iteration it shows B, B^-1, x_B = B^-1 b, the reduced costs, the ratio test
    and the update B^-1 <- E B^-1. Returns the final basis and the visited extreme points."""
    names, c, A, b = lp_data(prob)
    m, n = len(A), len(c)
    cols = names + [f"s{i+1}" for i in range(m)]
    M = [A[i] + [Fr(int(i == k)) for k in range(m)] for i in range(m)]      # [A | I]
    cost = c + [Fr(0)] * m
    basis = list(range(n, n + m))                                        # slacks basic: x = 0
    Binv = [[Fr(int(i == k)) for k in range(m)] for i in range(m)]
    path = []
    for it in range(100):
        xB = [sum(Binv[i][k] * b[k] for k in range(m)) for i in range(m)]
        x = [Fr(0)] * n
        for i, j in enumerate(basis):
            if j < n:
                x[j] = xB[i]
        path.append(x)
        cB = [cost[j] for j in basis]
        y = [sum(cB[i] * Binv[i][k] for i in range(m)) for k in range(m)]
        red = {j: cost[j] - sum(y[k] * M[k][j] for k in range(m)) for j in range(n + m) if j not in basis}
        z = sum(cb * v for cb, v in zip(cB, xB))
        B = [[M[i][j] for j in basis] for i in range(m)]
        if verbose:
            display(Markdown(f"#### Iteration {it}: extreme point x = ({', '.join(str(v) for v in x)}), z = {z}"))
            show(r"\text{basis} = (" + ", ".join(tex_var(cols[j]) for j in basis) + ")",
                 "B = " + tex_mat(B), "B^{-1} = " + tex_mat(Binv),
                 r"x_B = B^{-1} b = " + tex_col(xB), "z = c_B x_B = " + tex_num(z))
            show(r"y = c_B B^{-1} = (" + ", ".join(tex_num(v) for v in y) + ")",
                 r"\bar c_j = c_j - y A_j:\;" + r",\;".join(
                     rf"\bar c_{{{tex_var(cols[j])}}} = {tex_num(v)}" for j, v in red.items()))
        enter = max(red, key=lambda j: (red[j], -j))
        if red[enter] <= 0:
            if verbose:
                display(Markdown("**All reduced costs ≤ 0: no neighbour is better, this extreme point is optimal.**"))
            break
        d = [sum(Binv[i][k] * M[k][enter] for k in range(m)) for i in range(m)]
        ratios = {i: xB[i] / d[i] for i in range(m) if d[i] > 0}
        if not ratios:
            raise ValueError("unbounded LP")
        r = min(ratios, key=lambda i: (ratios[i], basis[i]))
        E = [[Fr(int(i == k)) for k in range(m)] for i in range(m)]
        for i in range(m):
            E[i][r] = 1 / d[r] if i == r else -d[i] / d[r]
        newBinv = matmul(E, Binv)
        if verbose:
            show(rf"{tex_var(cols[enter])}\text{{ enters (largest }}\bar c_j > 0)",
                 rf"d = B^{{-1}} A_{{{tex_var(cols[enter])}}} = " + tex_col(d),
                 r"\text{ratios } x_{B,i}/d_i:\;" + r",\;".join(
                     f"{tex_var(cols[basis[i]])}: {tex_num(v)}" for i, v in ratios.items()),
                 rf"\Rightarrow {tex_var(cols[basis[r]])}\text{{ leaves}}")
            show(r"B^{-1}_{\text{new}} = E\,B^{-1} = " + tex_mat(E) + tex_mat(Binv) + " = " + tex_mat(newBinv))
        Binv = newBinv
        basis[r] = enter
    return dict(basis=[cols[j] for j in basis], B=B, Binv=Binv, xB=xB, x=x, z=z, path=path,
                cols=cols, M=M, A=A, b=b, names=names, idx=basis)

# E: the explanation problem (used on the slides)
def make_E(integer=False):
    cat = "Integer" if integer else "Continuous"
    E = pulp.LpProblem("E", pulp.LpMaximize)
    x1 = E.add_variable("x1", lowBound=0, cat=cat)
    x2 = E.add_variable("x2", lowBound=0, cat=cat)
    E += 5 * x1 + 8 * x2                    # objective
    E += x1 + x2 <= 6, "c1"
    E += 5 * x1 + 9 * x2 <= 45, "c2"
    return E

# P: the practice problem
def make_P(integer=False):
    cat = "Integer" if integer else "Continuous"
    P = pulp.LpProblem("P", pulp.LpMaximize)
    x1 = P.add_variable("x1", lowBound=0, cat=cat)
    x2 = P.add_variable("x2", lowBound=0, cat=cat)
    P += 3 * x1 + 4 * x2
    P += x1 + 2 * x2 <= 8, "c1"
    P += 2 * x1 + x2 <= 9, "c2"
    return P

def frac(q):
    """Fractional part, always in [0, 1): frac(-1/4) = 3/4."""
    return q - math.floor(q)

def basis_report(prob, verbose=True):
    """Optimal basis of the LP relaxation of prob (exact), and every row  x_B(i) + sum_N abar_ij x_j = bbar_i."""
    res = simplex_steps(prob, verbose=False)
    M, cols, idx, Binv = res["M"], res["cols"], res["idx"], res["Binv"]
    m = len(M)
    nonbasic = [j for j in range(len(cols)) if j not in idx]
    tab = matmul(Binv, M)                                   # B^-1 [A | I]
    res["rows"] = [{"basic": cols[idx[i]], "coef": {cols[j]: tab[i][j] for j in nonbasic}, "rhs": res["xB"][i]}
                   for i in range(m)]
    res["nonbasic"] = [cols[j] for j in nonbasic]
    if verbose:
        lp = prob.deepcopy(); lp.solve(SOLVER)
        print("PuLP (floating point):", {v.name: round(v.value(), 6) for v in lp.variables()},
              " z =", round(pulp.value(lp.objective), 6))
        display(Markdown("**Exact optimal basis:**"))
        show(r"\text{basis} = (" + ", ".join(tex_var(r["basic"]) for r in res["rows"]) + ")",
             "B = " + tex_mat(res["B"]), "B^{-1} = " + tex_mat(Binv), "x_B = B^{-1}b = " + tex_col(res["xB"]),
             "z = " + tex_num(res["z"]))
        display(Markdown("**Rows of** $B^{-1}[A \\mid I]\\,x = B^{-1}b$ **(non-basic variables on the left):**"))
        for r in res["rows"]:
            show(tex_expr([1, *r["coef"].values()], [r["basic"], *r["coef"].keys()]) + " = " + tex_num(r["rhs"]))
    return res

def gomory_cut(prob, row=None, verbose=True):
    """Gomory fractional cut from one row of the optimal basis (default: f0 closest to 1/2).
    Returns (a, beta, basic) meaning the integer cut  a x <= beta, or None if the LP point is integer."""
    res = basis_report(prob, verbose=verbose)
    rows = [r for r in res["rows"] if frac(r["rhs"]) != 0]
    if not rows:
        if verbose:
            display(Markdown("**The LP solution is integer: no Gomory cut needed.**"))
        return None
    r = next(r for r in rows if r["basic"] == row) if row else min(rows, key=lambda r: abs(frac(r["rhs"]) - Fr(1, 2)))
    f = {j: frac(v) for j, v in r["coef"].items()}
    f0 = frac(r["rhs"])
    names, n, A, b = res["names"], len(res["names"]), res["A"], res["b"]
    m = len(A)
    # sum_j f_j x_j + sum_k f_{s_k} s_k >= f0   with   s_k = b_k - A_k x
    alpha = [sum(f.get(f"s{k+1}", 0) * A[k][j] for k in range(m)) - f.get(names[j], 0) for j in range(n)]
    beta = sum(f.get(f"s{k+1}", 0) * b[k] for k in range(m)) - f0
    L = math.lcm(*(q.denominator for q in alpha + [beta]))
    a_int, b_int = [int(q * L) for q in alpha], int(beta * L)
    g = math.gcd(*a_int, b_int) or 1
    a_int, b_int = [v // g for v in a_int], b_int // g
    if verbose:
        display(Markdown(f"**Gomory cut from the row of** ${tex_var(r['basic'])}$:"))
        show(r"f_j = \text{frac}(\bar a_j):\;" + r",\;".join(
            rf"\text{{frac}}({tex_num(v)}) = {tex_num(v)} - ({math.floor(v)}) = {tex_num(f[j])}"
            for j, v in r["coef"].items())
            + rf",\qquad f_0 = \text{{frac}}({tex_num(r['rhs'])}) = {tex_num(f0)}")
        show(r"\text{cut: }" + tex_expr(f.values(), f.keys()) + r" \ge " + tex_num(f0),
             r"\text{(at } x^* \text{ every non-basic variable is 0, so } 0 \ge " + tex_num(f0)
             + r"\text{ fails: } x^* \text{ is cut off)}")
        subs = [rf"s_{{{k+1}}} = {tex_num(b[k])} - ({tex_expr(A[k], names)})" for k in range(m) if f.get(f"s{k+1}", 0)]
        if subs:
            show(r"\text{substitute } " + r",\;".join(subs))
        final = r"\boxed{" + tex_expr(a_int, names) + r" \le " + str(b_int) + "}"
        if alpha == a_int and beta == b_int:
            show(r"\Rightarrow\;" + final)
        else:
            show(r"\Rightarrow\;" + tex_expr(alpha, names) + r" \le " + tex_num(beta),
                 r"\text{(times " + str(Fr(L, g)) + r")}\;\Rightarrow\;" + final)
        ga = math.gcd(*a_int)
        if ga > 1:
            display(Markdown(f"*Remark:* all coefficients are multiples of {ga}. Dividing by {ga} and rounding the "
                             f"right-hand side down gives the stronger cut "
                             f"${tex_expr([v // ga for v in a_int], names)} \\le {math.floor(Fr(b_int, ga))}$ "
                             "(a Chvátal–Gomory rounding, Part 5)."))
        print("GeoGebra:", ggb(a_int, b_int))
    return a_int, b_int, r["basic"]

def ggb(a, beta):
    """Inequality a x <= beta as GeoGebra input (x1 -> x, x2 -> y)."""
    terms = ""
    for coef, v in zip(a, ["x", "y"]):
        if coef:
            terms += ("-" if coef < 0 else ("+" if terms else "")) + ("" if abs(coef) == 1 else str(abs(coef))) + v
    return f"{terms} <= {beta}"

def gomory_loop(prob, max_cuts=10, verbose=True):
    """Pure cutting-plane loop: solve LP, add the Gomory cut, re-solve... (works on a copy of prob)."""
    p = prob.deepcopy()
    xs = p.variables()
    cuts, points = [], []
    for k in range(max_cuts + 1):
        res = simplex_steps(p, verbose=False)
        points.append(res["x"])
        if verbose:
            display(Markdown(f"### LP {k}: x* = ({', '.join(str(v) for v in res['x'])}), z = {res['z']}"))
        cut = gomory_cut(p, verbose=verbose)
        if cut is None:
            break
        a, beta, basic = cut
        cuts.append((a, beta, f"G{len(cuts) + 1}: " + ggb(a, beta).replace("x", "x1").replace("y", "x2")
                     + f"  (row {basic})"))
        p += pulp.lpSum(ai * v for ai, v in zip(a, xs)) <= beta, f"G{len(cuts)}"
    print("\nSummary:")
    for (a, beta, label), x in zip(cuts, points):
        print(f"  LP point ({', '.join(str(v) for v in x)}) -> {label}")
    print(f"  LP point ({', '.join(str(v) for v in points[-1])}) is integer: optimal.")
    print("GeoGebra (paste one line at a time):", *[ggb(a, beta) for a, beta, _ in cuts], sep="\n  ")
    return cuts, points

def cg_cut(prob, u, x_star=None):
    """Chvátal–Gomory cut  floor(u A) x <= floor(u b)  for multipliers u >= 0 (one per constraint)."""
    names, c, A, b = lp_data(prob)
    u = [Fr(v) for v in u]
    uA = [sum(u[i] * A[i][j] for i in range(len(A))) for j in range(len(names))]
    ub = sum(u[i] * b[i] for i in range(len(A)))
    a, beta = [math.floor(v) for v in uA], math.floor(ub)
    show(r"u^\top A\,x \le u^\top b:\;" + tex_expr(uA, names) + r" \le " + tex_num(ub),
         r"\Rightarrow\;" + tex_expr(a, names) + r" \le " + str(beta))
    if x_star is not None:
        lhs = sum(Fr(ai) * Fr(xi) for ai, xi in zip(a, x_star))
        print(f"at x* = {tuple(str(v) for v in x_star)}: lhs = {lhs} ->", "CUT OFF" if lhs > beta else "not cut off")
    return a, beta

## Live demo on $E$
**Step 1: the last basis and its equations.**

In [ ]:
E = make_E()
rep = basis_report(E)

**Step 2: one cut.** By default the row whose $f_0$ is closest to $\tfrac12$; try `row="x2"` too.

In [ ]:
gomory_cut(E);

**Step 3: the loop until the LP solution is integer**, and the picture (also paste the cuts in GeoGebra).

In [ ]:
cutsE, ptsE = gomory_loop(E)
plot_model(E, cuts=cutsE, star=ptsE[0], title="E: Gomory cuts"); plt.show()

**Things to notice.** Three cuts are needed starting from the $x_1$ row, but the $x_2$ row alone gives
$2x_1 + 3x_2 \le 15$, which makes the LP integer at once: **the choice of the row matters**.

## ⚠️ Numerical stability
Solvers work in floating point. A value that should be an integer comes back as `2.9999999999996`, and
`floor` turns that into a fractional part of almost 1:

In [ ]:
v = 3 - 4e-13                 # what a solver may report for x2 = 3
print("floor:", math.floor(v), " frac:", v - math.floor(v))
print("49 * (1/49) =", 49 * (1 / 49), " floor:", math.floor(49 * (1 / 49)))

With $f_0 \approx 1$ the row of an **integer** solution looks fractional, and its "cut"
$\sum_N f_j x_j \ge 0.9999\ldots$ is violated at the current vertex: it **cuts off an integer point, possibly the
optimum**. Wrong fractional parts on the coefficients give cuts that are simply invalid.

What real solvers do:
* treat $\text{frac}(a) < \varepsilon$ or $> 1 - \varepsilon$ as integer ($\varepsilon \approx 10^{-6}$);
* relax the right-hand side a little (safety margin) and discard cuts that are **dense** or whose coefficients
  range over many orders of magnitude (large *dynamism*);
* add only a few **rounds** of cuts, mostly at the root, and use the mixed-integer version (GMI);
* exact or *numerically safe* cuts exist (Cook, Dash, Fukasawa & Goycoolea, 2009) but cost more.

This notebook avoids the issue by computing $B^{-1}$ in exact fractions. That is fine for 2 variables, not for
10,000.

---
## ✍️ Activity 4 — Gomory on $P$
1. Run `basis_report(P)`: do you recognize $B^{-1}$ from Activity 2?
2. Run `gomory_loop(P)` and paste the cuts into GeoGebra (or look at the plot).
3. Answer on Miro (frame *4 – Gomory*):
   * the cuts, and the final LP solution;
   * does the **first** cut pass through any integer point? Look at its coefficients: what do you notice?
   * compare: how many B&B nodes (Activity 3) vs. how many cuts?

In [ ]:
P = make_P()
repP = basis_report(P)

In [ ]:
cutsP, ptsP = gomory_loop(P)
plot_model(P, cuts=cutsP, star=ptsP[0], title="P: Gomory cuts"); plt.show()

---
## ✍️ Activity 5 — Chvátal–Gomory: combine, then round
For multipliers $u \ge 0$ (one per constraint): $\;\lfloor u^\top A \rfloor x \le \lfloor u^\top b \rfloor$ is valid
for every integer point. Find the multipliers **on paper** first, then check with `cg_cut`.

1. On $E$: find $u$ that gives $2x_1 + 3x_2 \le 15$, and $u$ that gives $4x_1 + 7x_2 \le 35$.
   Compare with the rows of $B^{-1} = \tfrac14\begin{pmatrix}9 & -1\\ -5 & 1\end{pmatrix}$.
2. On $P$: find $u$ that gives $x_1 + x_2 \le 5$ in **one** step.
3. On $E$: $u = (0, \tfrac19)$ gives a valid inequality. Is it a cut for $x^* = (\tfrac94, \tfrac{15}4)$?

In [ ]:
xE = (Fr(9, 4), Fr(15, 4))
cg_cut(E, [Fr(1, 2), Fr(1, 2)], xE)        # example: u = (1/2, 1/2)
# ✍️ TODO: your multipliers, e.g. cg_cut(E, [Fr(?, ?), Fr(?, ?)], xE)

**Take-away.** Every Gomory cut is a CG cut, with $u$ = fractional parts of a row of $B^{-1}$. CG allows
**any** $u \ge 0$: more freedom, but finding a violated one is hard in general.

## 🚀 If you have time
Write your own small IP in PuLP (3 variables, 2–3 constraints, integer data, $b \ge 0$) and run
`gomory_loop` on it (`plot_model` works for 2 variables only). How many cuts? How large do the coefficients get?